In [1]:
import os
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
folders = ["doors_industrial", "fire_extinguisher_yolov8", "gauges_yolov8", "switches_panel"]

for f in folders:
    total_counts = Counter()
    for subset in ["train", "valid", "test"]:  # check all three
        label_dir = os.path.join(base, f, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        counts = Counter()
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, file)) as lab:
                for line in lab:
                    if line.strip():
                        counts[line.split()[0]] += 1
                        total_counts[line.split()[0]] += 1
        print(f"{f}/{subset} → class IDs: {dict(counts)}")
    print(f"Total for {f}: {dict(total_counts)}\n")


doors_industrial/train → class IDs: {'0': 652, '2': 745, '1': 645}
doors_industrial/valid → class IDs: {'0': 220, '2': 214, '1': 155}
doors_industrial/test → class IDs: {'0': 94, '2': 111, '1': 86}
➡️ Total for doors_industrial: {'0': 966, '2': 1070, '1': 886}

fire_extinguisher_yolov8/train → class IDs: {'0': 5608}
fire_extinguisher_yolov8/valid → class IDs: {'0': 606}
➡️ Total for fire_extinguisher_yolov8: {'0': 6214}

gauges_yolov8/train → class IDs: {'0': 388, '1': 595, '2': 339}
gauges_yolov8/valid → class IDs: {'0': 111, '1': 171, '2': 93}
gauges_yolov8/test → class IDs: {'0': 55, '1': 81, '2': 49}
➡️ Total for gauges_yolov8: {'0': 554, '1': 847, '2': 481}

switches_panel/train → class IDs: {'19': 337, '21': 941, '3': 272, '57': 474, '1': 238, '17': 156, '12': 3, '8': 588, '2': 313, '41': 76, '32': 66, '9': 92, '28': 14, '44': 2, '13': 359, '15': 49, '11': 23, '37': 67, '48': 126, '50': 88, '53': 14, '20': 14, '33': 4, '47': 29, '45': 12, '52': 9, '58': 4, '54': 3, '6': 8, '59': 

In [2]:
import os

# Folder base path
base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"

# Mapping dataset → new global ID
dataset_to_newid = {
    "fire_extinguisher_yolov8": 0,
    "switches_panel": 1,
    "doors_industrial": 2,
    "gauges_yolov8": 3,
}

for dataset, new_id in dataset_to_newid.items():
    print(f"\nFixing {dataset} → class ID {new_id}")
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, dataset, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            path = os.path.join(label_dir, file)
            with open(path, "r") as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 0:
                    continue
                parts[0] = str(new_id)
                new_lines.append(" ".join(parts))
            with open(path, "w") as f:
                f.write("\n".join(new_lines))
    print(f" {dataset} fixed.")



🔧 Fixing fire_extinguisher_yolov8 → class ID 0
✅ fire_extinguisher_yolov8 fixed.

🔧 Fixing switches_panel → class ID 1
✅ switches_panel fixed.

🔧 Fixing doors_industrial → class ID 2
✅ doors_industrial fixed.

🔧 Fixing gauges_yolov8 → class ID 3
✅ gauges_yolov8 fixed.


In [3]:
import os
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
folders = ["doors_industrial", "fire_extinguisher_yolov8", "gauges_yolov8", "switches_panel"]

for f in folders:
    total = Counter()
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, f, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, file)) as lab:
                for line in lab:
                    if line.strip():
                        total[line.split()[0]] += 1
    print(f"{f} → {dict(total)}")


doors_industrial → {'2': 2922}
fire_extinguisher_yolov8 → {'0': 6214}
gauges_yolov8 → {'3': 1882}
switches_panel → {'1': 6634}


In [5]:
from sklearn.model_selection import train_test_split
import glob, shutil, os

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
merged = os.path.join(base, "custom_merged_obb")

# create structure
for split in ["train", "valid"]:
    os.makedirs(os.path.join(merged, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(merged, f"labels/{split}"), exist_ok=True)

# collect all label paths
all_labels = []
for d in ["doors_industrial", "fire_extinguisher_yolov8", "gauges_yolov8", "switches_panel"]:
    for split in ["train", "valid"]:
        all_labels += glob.glob(os.path.join(base, d, split, "labels", "*.txt"))

train, val = train_test_split(all_labels, test_size=0.2, random_state=42)

for subset, files in [("train", train), ("valid", val)]:
    for label in files:
        img = label.replace("labels", "images").replace(".txt", ".jpg")
        if not os.path.exists(img):
            img = img.replace(".jpg", ".png")
        shutil.copy(label, os.path.join(merged, f"labels/{subset}"))
        shutil.copy(img, os.path.join(merged, f"images/{subset}"))

print("Merged dataset ready:", merged)


Merged dataset ready: E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb


In [ ]:
import os
from collections import Counter

base = r"E:\sem7\FYP\9_24\objectdetetction\datasets"
folders = ["doors_industrial", "fire_extinguisher_yolov8", "gauges_yolov8", "switches_panel"]

for f in folders:
    total = Counter()
    for subset in ["train", "valid", "test"]:
        label_dir = os.path.join(base, f, subset, "labels")
        if not os.path.exists(label_dir):
            continue
        for file in os.listdir(label_dir):
            if not file.endswith(".txt"):
                continue
            with open(os.path.join(label_dir, file)) as lab:
                for line in lab:
                    if line.strip():
                        total[line.split()[0]] += 1
    print(f"{f} → {dict(total)}")


doors_industrial → {'2': 2922}
fire_extinguisher_yolov8 → {'0': 6214}
gauges_yolov8 → {'3': 1882}
switches_panel → {'1': 6634}


In [1]:
# --------------------------------------------------------------
# train_finetune.py — memory-safe YOLOv8-OBB fine-tuning on GTX 1660 Ti (6 GB)
# --------------------------------------------------------------
from ultralytics import YOLO
import torch, gc

if __name__ == "__main__":
    # Clear cache
    gc.collect()
    torch.cuda.empty_cache()

    # Load pretrained YOLOv8 OBB model
    model = YOLO("yolov8s-obb.pt")

    # Start fine-tuning
    model.train(
        data=r"E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb\data.yaml",
        epochs=100,
        imgsz=640,           # 416px to fit 6 GB GPU
        batch=16,             # safe batch size for GTX 1660 Ti
        device=0,            # use first GPU
        workers=0,           # prevent Windows DataLoader crashes
        amp=True,            # mixed precision (reduces VRAM use)
        mosaic=0.0,          # disable mosaic (stable on OBB data)
        mixup=0.0,           # disable mixup
        erasing=0.0,         # no random erasing
        cos_lr=False,
        lr0=0.001,
        optimizer="SGD",
        name="yolov8s_custom_merged_obb_vramfix",
        patience=10,
        verbose=True,
        deterministic=True,  # same results each run
    )

    # Evaluate after training
    metrics = model.val()
    print(metrics)


New https://pypi.org/project/ultralytics/8.3.209 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.10.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\sem7\FYP\9_24\objectdetetction\datasets\custom_merged_obb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-obb.pt, momentum=0.937, mosaic=0.0, m

KeyboardInterrupt: 